In [1]:
import yaml

from pydrake.all import (
    StartMeshcat,
    AddDefaultVisualization,
    Simulator,
    RobotDiagramBuilder,
    VPolytope,
    HPolyhedron,
    SceneGraphCollisionChecker,
    RandomGenerator,
    PointCloud,
    Rgba,
    Quaternion,
    RigidTransform,
    IrisFromCliqueCoverOptions,
    IrisInConfigurationSpaceFromCliqueCoverV2,
    SaveIrisRegionsYamlFile,
    LoadModelDirectives,
    ProcessModelDirectives,
    CollisionCheckerParams,
    Sphere,
    GaussianVectorX,
    RandomGenerator,
    UniformVector,
    IrisZoFromCliqueBuilder,
    MinCliqueCoverSolverViaGreedy,
    MaxCliqueSolverViaGreedy,
    Parallelism,
    PointsToCliqueCoverSets,
    VisibilityGraph,
    HPolyhedronPointSampler,
    InverseKinematics,
Solve,
ModelInstanceInfo
)
import numpy as np
import os
import data.iris_benchmarks.benchmarks.helpers as benchmark_helpers
from data.iris_benchmarks.benchmarks.helpers import load_seed_points
from data.iris_benchmarks.iris_environments.environments import get_environment_builder
import networkx as nx
import tqdm
import time
import pathlib
import utils
import matplotlib.pyplot as plt

prop_cycle = plt.rcParams['axes.prop_cycle']
colors = prop_cycle.by_key()['color']
import functools

In [2]:
TEST_SCENE = "BOXUNLOADING"
EXPERIMENT_NAME = "rrt_seeding"

src_directory = os.path.abspath(os.path.abspath(os.path.join(os.getcwd(), os.pardir)))
parent_directory = os.path.dirname(src_directory)
data_directory = os.path.join(parent_directory, "data")

experiment_data_folder = os.path.join(data_directory, TEST_SCENE, EXPERIMENT_NAME)
pathlib.Path(experiment_data_folder).mkdir(parents=True, exist_ok=True)
region_file = os.path.join(experiment_data_folder, "iris_regions.yaml")

meshcat = StartMeshcat()
robot_diagram_builder = RobotDiagramBuilder()
parser = robot_diagram_builder.parser()
iris_environement_assets = os.path.join(data_directory, "iris_benchmarks_scenes_urdf", "iris_environments", "assets")
parser.package_map().Add("iris_environments",iris_environement_assets)
from scenario import scenario_yaml_for_iris
model_indices = parser.AddModelsFromString(scenario_yaml_for_iris, ".dmd.yaml")
plant = robot_diagram_builder.plant()
plant.Finalize()
scene_graph = robot_diagram_builder.scene_graph()
AddDefaultVisualization(robot_diagram_builder.builder(), meshcat=meshcat)
diagram = robot_diagram_builder.Build()
diagram_context = diagram.CreateDefaultContext()
plant_context = plant.GetMyMutableContextFromRoot(diagram_context)
diagram.ForcedPublish(diagram_context)
print("Model Names:")
for model_index in model_indices:
    print(f"{model_index}: {plant.GetModelInstanceName(model_index)}")
        

INFO:drake:Meshcat listening for connections at http://localhost:7000


Model Names:
ModelInstanceIndex(2): robot_base
ModelInstanceIndex(3): kuka
ModelInstanceIndex(4): Truck_Trailer_Floor
ModelInstanceIndex(5): Truck_Trailer_Right_Side
ModelInstanceIndex(6): Truck_Trailer_Left_Side
ModelInstanceIndex(7): Truck_Trailer_Roof
ModelInstanceIndex(8): Truck_Trailer_Back


In [3]:
collision_checker = SceneGraphCollisionChecker(model = diagram,
                                               robot_model_instances = model_indices,
                                               edge_step_size = 0.02)

INFO:drake:Allocating contexts to support implicit context parallelism 16


In [4]:
import yaml
seed_point_file = os.path.join(data_directory, "pickle_data", "seedpoints.yml")
with open(seed_point_file, "r") as f:
    seed_points = yaml.safe_load(f)
seed_points = np.array(seed_points["seedpoints"])
print(seed_points)

[[-6.0000e-04 -9.1230e-01  1.0901e+00  1.6000e-03  1.3930e+00 -9.0000e-04]
 [ 1.4210e-01 -9.5270e-01  1.2014e+00  1.0913e+00  1.3148e+00 -1.5310e-01]
 [ 1.4770e-01 -1.4674e+00  1.7249e+00  1.7200e-02  6.7650e-01 -2.7300e-02]
 [-1.3000e-03 -2.5138e+00  2.4823e+00 -8.0000e-04  1.6113e+00 -0.0000e+00]
 [-3.1010e-01 -1.4551e+00  1.1829e+00 -1.7670e-01  1.1339e+00 -0.0000e+00]]


In [5]:
seed_points.shape

(5, 6)

In [6]:

end_effector_model_index = model_indices[1]
print("End Effector Bodies:")
for body_index in plant.GetBodyIndices(end_effector_model_index):
    print(plant.get_body(body_index).name())
end_effector = plant.GetBodyByName("arm_eef", end_effector_model_index)
end_effector_frame_id = plant.GetBodyFrameIdOrThrow(end_effector.index())


    
def get_end_effector_pose(q, context=None):
    utils.get_end_effector_pose(q, end_effector, plant, context, model_index=end_effector_model_index)


def plot_end_effector_configurations(
    configs, meshcat, name, radius=0.05, color=Rgba(1, 0, 0, 1)
):
    utils.plot_end_effector_configurations(
        configs, end_effector, plant, meshcat, name, radius=0.05, color=Rgba(1, 0, 0, 1), model_index = end_effector_model_index
    )


def plot_end_effector_configurations_as_point_cloud(
    configs, meshcat, name, radius=0.01, color=Rgba(1, 0, 0, 1)
):
    utils.plot_end_effector_configurations_as_point_cloud(
        configs, end_effector, plant, meshcat, name, radius, color, end_effector_model_index
    )


plot_end_effector_configurations(seed_points, meshcat, "seed_points")

End Effector Bodies:
base_link
arm_base
arm_a1
arm_a2
arm_a3
arm_a4
arm_a5
arm_a6
arm_eef


In [7]:
class SphereicalRRT:
    def __init__(self, collision_checker, random_generator):
        self.collision_checker = collision_checker
        self.dim = self.collision_checker.plant().num_positions()
        self.random_generator = random_generator
        self.gaussian = GaussianVectorX(np.zeros(self.dim), np.ones(self.dim))
        self.uniform = UniformVector(np.zeros(1), np.ones(1))

    def sample_uniformly_in_sphere(self, seed_point, sphere_radius):
        direction = self.gaussian.Sample(self.random_generator)
        direction /= np.linalg.norm(direction)
        r = sphere_radius * self.uniform.Sample(self.random_generator).item()
        return r ** (1 / self.dim) * direction + seed_point

    def find_closest_point_in_tree(self, tree, point):
        point_distances_pairs = [
            (q, self.collision_checker.ComputeConfigurationDistance(np.array(q), point))
            for q in tree
        ]
        return np.array(min(point_distances_pairs, key=lambda x: x[1])[0])

    def extend(self, start_point, end_point):
        """
        Extend as far from start_point towards end_point and return the configuration
        """
        edge_measure = self.collision_checker.MeasureEdgeCollisionFree(
            start_point, end_point
        )
        assert edge_measure.partially_free()
        return self.collision_checker.InterpolateBetweenConfigurations(
            start_point, end_point, edge_measure.alpha()
        )

    def add_next_point(self, next_direction, tree):
        closest_point = self.find_closest_point_in_tree(tree, next_direction)
        next_point = self.extend(closest_point, next_direction)
        closest_point_tuple = tuple(closest_point)
        next_point_tuple = tuple(next_point)
        tree.add_node(next_point_tuple)
        tree.add_edge(closest_point_tuple, next_point_tuple)

    def build(
        self, seed_point, num_points, radius, goal=None, sample_goal_fraction=0.1
    ):
        assert seed_point.shape == (self.dim,)
        if goal is not None:
            assert goal.shape == seed_point.shape
        tree = nx.Graph()
        tree.add_node(tuple(seed_point))
        
        # pbar = tqdm.tqdm(desc="Building RRT", total=num_points)
        ctr = 0
        while tree.number_of_nodes() < num_points:
            if goal is not None and np.random.rand() < sample_goal_fraction:
                next_direction = goal
            else:
                next_direction = self.sample_uniformly_in_sphere(seed_point, radius)
            self.add_next_point(next_direction, tree)
            # pbar.update(tree.number_of_nodes())
            ctr += 1
        # pbar.close()
        return tree



rrt_builder = SphereicalRRT(collision_checker, RandomGenerator(seed=0))


In [8]:
seed_point_index = 4
rrt_points = {}
seed_point_colors = {}
for i, seed_point in tqdm.tqdm(enumerate(seed_points)):
    if i == seed_point_index:
        tree = rrt_builder.build(
            seed_point=seed_point,
            num_points=1000,
            radius=1,
            goal=None,
            sample_goal_fraction=0.1,
        )
        rrt_points[i] = np.array(tree.nodes)
        color = Rgba(*utils.hex_to_rgb_0_1(colors[i]), 1)
        plot_end_effector_configurations_as_point_cloud(rrt_points[i], meshcat, f"seed_point_{i}/rrt_points", radius=0.01, color=color)
        plot_end_effector_configurations(seed_point[np.newaxis, :], meshcat, f"seed_point_{i}", radius=0.1, color=color)
    

5it [00:03,  1.57it/s]


In [9]:
from pydrake.all import IrisZoOptions

zo_options = IrisZoOptions()
zo_options.configuration_space_margin = 0.01
zo_options.num_particles = int(1e3)
zo_options.max_separating_planes_per_iteration = 5
zo_options.epsilon = 0.01
zo_options.max_iterations = 1
zo_options.verbose = True

set_builder = IrisZoFromCliqueBuilder(collision_checker, options=zo_options)
max_clique_solver = MaxCliqueSolverViaGreedy()
min_clique_cover_solver = MinCliqueCoverSolverViaGreedy(max_clique_solver)
min_clique_cover_solver.set_min_clique_size(20)

points = rrt_points[seed_point_index].T
# sets = PointsToCliqueCoverSets(points, collision_checker, min_clique_cover_solver, set_builder)


In [10]:
t0 = time.time()
visibility_graph = VisibilityGraph(collision_checker, points, Parallelism.Max())
t1 = time.time()
print(f"Visibilty Graph took {t1-t0} seconds")

Visibilty Graph took 70.35795474052429 seconds


In [11]:
t0 = time.time()
clique_cover = min_clique_cover_solver.SolveMinCliqueCover(visibility_graph, True)
t1 = time.time()
print(f"Clique Cover took {t1-t0} seconds")
print(f"There are {len(clique_cover)} cliques")

Clique Cover took 0.061653852462768555 seconds
There are 4 cliques


In [12]:
import pickle

points_v_graph = (points, visibility_graph, clique_cover)
with open(os.path.join(experiment_data_folder, f"rrt_tree_{seed_point_index}_points_and_v_graph.pkl"), "wb") as f:
    pickle.dump(points_v_graph, f)


In [13]:
for clique in clique_cover:
    print(len(clique))

717
44
47
26


In [14]:
sets = []
for clique_inds in clique_cover:
    clique = np.zeros((points.shape[0],len(clique_inds)))
    for i, ind in enumerate(clique_inds):
        clique[:,i] = points[:,ind]
    sets.append(set_builder.BuildRegion(clique))
    # break

INFO:drake:FastIris finding region that is 0.01 collision free with 0.95 certainty using 1000 particles.
INFO:drake:FastIris worst case test requires 7986 samples.
INFO:drake:FastIris iteration 0
INFO:drake:FastIris N_k 2795, N_col 2304, thresh 13.975
INFO:drake:SeparatingPlanes iteration: 1 faces: 17
INFO:drake:FastIris N_k 3904, N_col 3218, thresh 19.52
INFO:drake:FastIris N_k 4553, N_col 3496, thresh 22.765
INFO:drake:FastIris N_k 5013, N_col 605, thresh 25.065
INFO:drake:FastIris N_k 5370, N_col 684, thresh 26.85
INFO:drake:FastIris N_k 5662, N_col 794, thresh 28.310000000000002
INFO:drake:FastIris N_k 5908, N_col 508, thresh 29.54
INFO:drake:FastIris N_k 6122, N_col 432, thresh 30.61
INFO:drake:FastIris N_k 6310, N_col 471, thresh 31.55
INFO:drake:FastIris N_k 6479, N_col 391, thresh 32.395
INFO:drake:FastIris N_k 6631, N_col 397, thresh 33.155
INFO:drake:FastIris N_k 6771, N_col 84, thresh 33.855000000000004
INFO:drake:FastIris N_k 6899, N_col 18, thresh 34.495
INFO:drake:FastIri

In [15]:
regions_dict = {f"set{i}": s for i, s in enumerate(sets)}
SaveIrisRegionsYamlFile(os.path.join(experiment_data_folder, f"rrt_tree_{seed_point_index}_regions.pkl"), regions_dict)

In [21]:
def visualize_region(region, meshcat, name, num_samples = 1000,  radius=0.01, color=Rgba(1, 0, 0, 1)):
    utils.visualize_region(region, end_effector,
                           plant, meshcat, name, num_samples, radius, color)
    
for idx, set in enumerate(sets):
    color = Rgba(*utils.hex_to_rgb_0_1(colors[idx]), 1)
    visualize_region(sets[idx], meshcat, f"seed_point_{seed_point_index}/set_{idx}", color = color) 
    

In [ ]:
set_builder = IrisZoFromCliqueBuilder(collision_checker, options = zo_options)
max_clique_solver = MaxCliqueSolverViaGreedy()
min_clique_cover_solver = MinCliqueCoverSolverViaGreedy(max_clique_solver)
min_clique_cover_solver.set_min_clique_size(20)
sets = []
for seed_point_index, points in rrt_points.items():
    points = points.T
    print(f"Starting Seed Point {seed_point_index+1}/{len(rrt_points)}")
    t0 = time.time()
    visibility_graph = VisibilityGraph(collision_checker, points, Parallelism.Max())
    t1 = time.time()
    print(f"Visibilty Graph took {t1-t0} seconds")
    
    t0 = time.time()
    clique_cover = min_clique_cover_solver.SolveMinCliqueCover(visibility_graph, True)
    t1 = time.time()
    print(f"Clique Cover took {t1-t0} seconds")
    print(f"There are {len(clique_cover)} cliques")
    print("Clique sizes =")
    for clique in clique_cover:
        print(len(clique))
    new_sets = []
    for clique_inds in clique_cover:
        clique = np.zeros((points.shape[0],len(clique_inds)))
        for i, ind in enumerate(clique_inds):
            clique[:,i] = points[:,ind]
        new_sets.append(set_builder.BuildRegion(clique))
    sets.append(new_sets)

Starting Seed Point 1/10
Visibilty Graph took 177.84153127670288 seconds
Clique Cover took 0.04977607727050781 seconds
There are 1 cliques
Clique sizes =
993


INFO:drake:FastIris iteration 0
INFO:drake:FastIris iter 1, iter limit 1


Starting Seed Point 2/10


INFO:drake:FastIris iteration 0


Visibilty Graph took 133.6439332962036 seconds
Clique Cover took 0.03203845024108887 seconds
There are 7 cliques
Clique sizes =
297
47
42
39
107
26
35


...

Starting Seed Point 3/10


INFO:drake:FastIris iteration 0
[*** LOG ERROR #0001 ***] [2024-11-08 14:29:08] [console] KeyboardInterrupt: <EMPTY MESSAGE>

At:
  /usr/lib/python3.12/logging/__init__.py(1790): isEnabledFor
  /tmp/ipykernel_294112/3348319154.py(27): <module>
  /home/amice/Documents/coding_projects/GCS-Box-Unloading/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py(3577): run_code
  /home/amice/Documents/coding_projects/GCS-Box-Unloading/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py(3517): run_ast_nodes
  /home/amice/Documents/coding_projects/GCS-Box-Unloading/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py(3334): run_cell_async
  /home/amice/Documents/coding_projects/GCS-Box-Unloading/.venv/lib/python3.12/site-packages/IPython/core/async_helpers.py(128): _pseudo_sync_runner
  /home/amice/Documents/coding_projects/GCS-Box-Unloading/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py(3130): _run_cell
  /home/amice/Documents/cod

Visibilty Graph took 131.1099569797516 seconds
Clique Cover took 0.021413087844848633 seconds
There are 4 cliques
Clique sizes =
281
79
53
23


INFO:drake:FastIris iteration 0
INFO:drake:FastIris delta vol 0.004425019482120733, threshold 0.02
INFO:drake:FastIris iteration 0
INFO:drake:FastIris iter 1, iter limit 1
INFO:drake:FastIris iteration 0
INFO:drake:FastIris delta vol 0.0011528602830500503, threshold 0.02


Starting Seed Point 4/10


In [17]:
sets

[[<pydrake.geometry.optimization.HPolyhedron at 0x7838c007ff30>],

In [18]:
regions_dict = {f"tree_{i}_set_{j}": s for i, set_i in enumerate(sets) for j, s in enumerate(set_i)}
SaveIrisRegionsYamlFile(region_file, regions_dict)
for seed_point_index, set_i in enumerate(sets):
    for j, s in enumerate(set_i):
        color = Rgba(*hex_to_rgb_0_1(colors[(seed_point_index+j)%len(colors)]), 1)
        visualize_region(s, meshcat, f"seed_point_{seed_point_index}/set_{j}", color = color) 

In [19]:
regions_dict


{'tree_0_set_0': <pydrake.geometry.optimization.HPolyhedron at 0x7838c007ff30>,
 'tree_1_set_0': <pydrake.geometry.optimization.HPolyhedron at 0x7838a3914670>,
 'tree_1_set_1': <pydrake.geometry.optimization.HPolyhedron at 0x78388a5257b0>,
 'tree_1_set_2': <pydrake.geometry.optimization.HPolyhedron at 0x7838c3f8b930>,
 'tree_1_set_3': <pydrake.geometry.optimization.HPolyhedron at 0x78388a5245f0>,
 'tree_1_set_4': <pydrake.geometry.optimization.HPolyhedron at 0x78388a526f30>,
 'tree_1_set_5': <pydrake.geometry.optimization.HPolyhedron at 0x78388a5275b0>,
 'tree_1_set_6': <pydrake.geometry.optimization.HPolyhedron at 0x7838c03d9930>,
 'tree_2_set_0': <pydrake.geometry.optimization.HPolyhedron at 0x78388a5173f0>,
 'tree_2_set_1': <pydrake.geometry.optimization.HPolyhedron at 0x7838a39f62b0>,
 'tree_2_set_2': <pydrake.geometry.optimization.HPolyhedron at 0x7838c017e7b0>,
 'tree_2_set_3': <pydrake.geometry.optimization.HPolyhedron at 0x7838c01fe0b0>,
 'tree_3_set_0': <pydrake.geometry.optim

In [23]:
sampler = HPolyhedronPointSampler(HPolyhedron.MakeBox(plant.GetPositionLowerLimits(), plant.GetPositionUpperLimits()), 100)
num_points = int(1e5)
t0 = time.time()
test_points = sampler.SamplePoints(num_points, RandomGenerator(0))
t1 = time.time()
print(f"Sampling {num_points} took {t1-t0}s")

Sampling 100000 took 4.178552627563477s


In [32]:
collision_free_inds = np.array(collision_checker.CheckConfigsCollisionFree(test_points.T)).astype(bool)
collision_free_points = test_points[:, collision_free_inds]
print(f"{collision_free_points.shape[1]}/{test_points.shape[1]} collision free points")

48107/100000 collision free points


In [35]:
all_sets = [s for set_list in sets for s in set_list]

In [36]:
# the (i,j) entry of this array is true if the jth point is in the ith set
points_in_sets_mask = np.array([[s.PointInSet(test_points[:,j]) for j in range(test_points.shape[1])] for s in all_sets])


In [42]:
freq_point_in_set = points_in_sets_mask.sum(axis = 0)
points_in_sets = test_points[:, freq_point_in_set >= 1]
print(f"{points_in_sets.shape[1]}/{test_points.shape[1]} total points in sets")
print(f"{points_in_sets.shape[1]}/{collision_free_points.shape[1]} collision free points in sets")

array([0, 0, 0, ..., 0, 0, 0])

In [131]:
def ik_via_sets(task_space_point, sets):
    world_frame = plant.GetFrameByName("world")

    configs_ret = []
    for i, s in enumerate(sets):
        ik = InverseKinematics(plant)
        ik.AddPositionConstraint(
            end_effector.body_frame(),
            np.zeros(3),
            world_frame,
            task_space_point,
            task_space_point,
        )
        variables, point_in_set_constraints = s.AddPointInSetConstraints(
            ik.get_mutable_prog(), ik.q()
        )
        ik.get_mutable_prog().SetInitialGuess(ik.q(), s.ChebyshevCenter())
        result = Solve(ik.prog())
        if not result.is_success():
            configs_ret.append(None)
        else:
            configs_ret.append(result.GetSolution(ik.q()))

    return configs_ret


test_config = seed_points[1]
test_task_space_point = get_end_effector_pose(test_config)
test_configs = ik_via_sets(test_task_space_point[:, np.newaxis], all_sets)
test_configs
        
    

[None,
 None,
 None,
 None,
 array([-1.24032963, -0.68242764,  0.23880007,  0.33096974,  1.21300905,
        -1.06090405,  0.63617298]),
 None,
 array([-1.26522054, -0.5360749 ,  0.29248316,  0.57971735,  1.13011962,
        -0.82510192,  0.45660339]),
 None,
 array([-1.04338488, -0.3506186 ,  0.06702371,  0.97268791,  2.20622134,
        -0.25557321,  2.28765622]),
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 array([-0.57799952, -0.84333956,  1.02830053, -0.58888406,  2.08839741,
         1.09392958,  2.226877  ]),
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 array([-0.72836627, -0.35889235,  2.86363829, -0.85302285,  2.70941868,
        -0.24533371,  0.42546074]),
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 array([-0.93800723, -0.43854159,  2.96705577, -0.97723357,  2.57878801,
         0.70055361,  0.96633029]),
 None,
 None,
 None,
 None,
 None]

In [130]:
# test_config = seed_points[1]
# test_task_space_point = get_end_effector_pose(test_config)
# world_frame = plant.GetFrameByName("world")
# 
# ik = InverseKinematics(plant)
# ik.AddPositionConstraint(
#     end_effector.body_frame(),
#     np.zeros(3),
#     world_frame,
#     test_task_space_point,
#     test_task_space_point,
# )
# # variables, point_in_set_constraints = s.AddPointInSetConstraints(
# #     ik.get_mutable_prog(), ik.q()
# # )
# 
# ik.get_mutable_prog().SetInitialGuess(ik.q(), s.ChebyshevCenter())
# result = Solve(ik.prog())
# print(f"{result.is_success()=}")
# q_solved = result.GetSolution(ik.q())
# print(f"{test_task_space_point=}")
# task_space_point_achieved = get_end_effector_pose(q_solved)
# print(f"{task_space_point_achieved=}")


result.is_success()=False
test_task_space_point=array([-0.41175491,  0.56605187,  0.88922587])
task_space_point_achieved=array([-0.41175491,  0.48058064,  0.88922587])


In [149]:
task_space_hpolyhedron = HPolyhedronPointSampler(
    HPolyhedron.MakeBox(np.array([-0.75, -1, 0.1]), np.array([-0.25, 1, 1])), 100
)
num_task_space_points = int(1000)
task_space_points = task_space_hpolyhedron.SamplePoints(
    num_task_space_points, RandomGenerator(0)
).T
utils.view_points_as_point_cloud(task_space_points, meshcat, "task_space_points")


SystemExit: Failure at bazel-out/k8-opt/bin/multibody/plant/_virtual_includes/_multibody_plant_core_headers_cc_impl/drake/multibody/plant/multibody_plant.h:2642 in SetPositions(): condition 'q.size() == num_positions()' failed.

In [140]:
task_space_points_configs = []
for i, task_space_point in tqdm.tqdm(enumerate(task_space_points)):
    task_space_points_configs.append(ik_via_sets(task_space_point, all_sets))


1000it [16:05,  1.04it/s]


In [143]:
task_space_point_is_reachable = np.array([
    not all(item is None for item in configs) for configs in task_space_points_configs
])
example_reaching_config = [
   next((item for item in configs if item is not None), None) for configs in task_space_points_configs
]
print(task_space_point_is_reachable.sum())

189


In [169]:
for config in example_reaching_config:
# for config in task_space_points_configs[0]:
    if config is not None:
        # print(get_end_effector_pose(config))
        plant.SetPositions(plant_context, config)
        diagram.ForcedPublish(diagram_context)
        time.sleep(0.5)

In [167]:
from pydrake.all import MinimumDistanceLowerBoundConstraint
def ik_via_sets_with_fallback(task_space_point, sets):
    world_frame = plant.GetFrameByName("world")
    context = diagram.GetMutableSubsystemContextFromRoot(plant)

    configs_ret = []
    config_found_in_set = []
    for i, s in enumerate(sets):
        ik = InverseKinematics(plant, context)
        ik.AddPositionConstraint(
            end_effector.body_frame(),
            np.zeros(3),
            world_frame,
            task_space_point,
            task_space_point,
        )
        variables, point_in_set_constraints = s.AddPointInSetConstraints(
            ik.get_mutable_prog(), ik.q()
        )
        ik.get_mutable_prog().SetInitialGuess(ik.q(), s.ChebyshevCenter())
        result = Solve(ik.prog())
        if not result.is_success():
            for c in ik.prog().GetAllConstraints():
                ik.get_mutable_prog().RemoveConstraint(c)
            ik.get_mutable_prog().AddConstraint(MinimumDistanceLowerBoundConstraint(plant, 1e-4, context))
            result = Solve(ik.prog())
            if not result.is_success():
                configs_ret.append(None)
                config_found_in_set.append(False)
            else:
                configs_ret.append(result.GetSolution(ik.q()))
                config_found_in_set.append(False)
        else:
            configs_ret.append(result.GetSolution(ik.q()))
            config_found_in_set.append(True)

    return configs_ret, config_found_in_set

In [168]:
print(task_space_points[:,0])
configs_ret_fallback, configs_found_in_set_fallback = ik_via_sets_with_fallback(task_space_points[:,0], all_sets)
print(configs_ret_fallback)

[-0.29979518 -0.61404076  0.15588041]


AttributeError: 'pydrake.planning.RobotDiagram' object has no attribute 'GetMutableSubsystemContextFromRoot'

In [173]:
from iris import IrisRegionGenerator
IrisRegionGenerator.visualize_connectivity(all_sets, 0.7, False)

<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: IRIS region connectivity Pages: 1 -->
<svg width="5225pt" height="2780pt"
 viewBox="0.00 0.00 5225.28 2780.00" xmlns="http://www.w3.org/2000/svg" xmlns:xlink="http://www.w3.org/1999/xlink">
<g id="graph0" class="graph" transform="scale(1 1) rotate(0) translate(4 2776)">
<title>IRIS region connectivity</title>
<polygon fill="white" stroke="transparent" points="-4,4 -4,-2776 5221.28,-2776 5221.28,4 -4,4"/>
<!-- 0 -->
<g id="node1" class="node">
<title>0</title>
<ellipse fill="none" stroke="black" cx="2693.22" cy="-2754" rx="27" ry="18"/>
<text text-anchor="middle" x="2693.22" y="-2750.3" font-family="Times,serif" font-size="14.00">0</text>
</g>
<!-- 1 -->
<g id="node2" class="node">
<title>1</title>
<ellipse fill="none" stroke="black" cx="2389.22" cy="-2682" rx="27" ry="18"

(49, 383)

"black" d="M1207.7,-736.35C1257.25,-733.86 1345.58,-723.66 1408.22,-684 1596.61,-564.73 1528.69,-408.61 1716.22,-288 1774.1,-250.78 1855.53,-239.66 1902.3,-236.36"/>
<polygon fill="black" stroke="black" points="1207.29,-732.86 1197.45,-736.78 1207.59,-739.86 1207.29,-732.86"/>
<polygon fill="black" stroke="black" points="1902.52,-239.85 1912.28,-235.74 1902.09,-232.87 1902.52,-239.85"/>
</g>
<!-- 36&#45;&gt;46 -->
<g id="edge344" class="edge">
<title>36&#45;&gt;46</title>
<path fill="none" stroke="black" d="M1207.16,-733.42C1272.98,-726.47 1407.3,-709.56 1446.22,-684 1583.38,-593.94 1654.22,-543.08 1654.22,-379 1654.22,-379 1654.22,-379 1654.22,-305 1654.22,-216.49 1242.58,-323.41 1903.22,-216 2115.87,-181.43 3715.26,-165.71 3980.89,-163.32"/>
<polygon fill="black" stroke="black" points="1206.42,-729.98 1196.84,-734.49 1207.14,-736.94 1206.42,-729.98"/>
<polygon fill="black" stroke="black" points="3981.19,-166.82 3991.16,-163.23 3981.13,-159.82 3981.19,-166.82"/>
</g>
<!-- 36&#45;&gt;4